# Semantic Search using PineCone DB (Cloud Based)

website: https://www.pinecone.io/

## Objective

1) Convert text → embeddings

2) Store in vector database

3) Perform semantic search

In [1]:
# Following too long time
!pip install pinecone sentence-transformers

# SO I used 
# !python -m !pip install pinecone sentence-transformers

  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
   ---------------------------------------- 0.0/742.7 kB ? eta -:--:--
   ---------------------------------------- 742.7/742.7 kB 7.6 MB/s  0:00:00
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl (6.2 kB)

   ---------------------------------------- 0/5 [pinecone-plugin-interface]
  Attempting uninstall: packaging
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
    Found existing installation: packaging 25.0
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
    Uninstalling packaging-25.0:
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
      Successfully uninstalled packaging-25.0
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
   -------- ---------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# Import Librarairs: Took a 4-5 minutes
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [ ]:
# Need API Key
pc = Pinecone(api_key="pcsk_API_KEY_HERE")

# Create an index

In [43]:
index_name = "genai-demo"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

# Connect to Index

In [44]:
index = pc.Index(index_name)

# Prepare the documents

In [45]:
documents = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Python is used for data science",
    "Cars and vehicles are transportation",
    "Artificial intelligence is transforming industries",
    "Football is a popular sport",
    "Data engineering involves pipelines",
    "Big data tools include Hadoop and Spark",
    "Chatbots are powered by large language models",
    "Healthcare is improved by AI diagnostics. It is bringing positive change in the world",
    "Self-driving cars use machine learning"
]

In [46]:
vectors = [
    {
        "_id": str(i),
        "chunk_text": doc   # must match field_map
    }
    for i, doc in enumerate(documents)
]

print(vectors)

[{'_id': '0', 'chunk_text': 'Machine learning is a subset of AI'}, {'_id': '1', 'chunk_text': 'Deep learning uses neural networks'}, {'_id': '2', 'chunk_text': 'Python is used for data science'}, {'_id': '3', 'chunk_text': 'Cars and vehicles are transportation'}, {'_id': '4', 'chunk_text': 'Artificial intelligence is transforming industries'}, {'_id': '5', 'chunk_text': 'Football is a popular sport'}, {'_id': '6', 'chunk_text': 'Data engineering involves pipelines'}, {'_id': '7', 'chunk_text': 'Big data tools include Hadoop and Spark'}]


# Store the documents in vector DB

In [47]:
# Upsert the records into a namespace
index.upsert_records("example-namespace", vectors)

UpsertResponse(upserted_count=8, _response_info={'raw_headers': {'date': 'Thu, 19 Mar 2026 01:53:52 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '348', 'x-pinecone-response-duration-ms': '349', 'server': 'envoy'}})

In [48]:
# Wait for the upserted vectors to be indexed
import time
time.sleep(10)

# View stats for the index
stats = index.describe_index_stats()
print(stats)

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '188',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 01:53:57 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '50',
                                    'x-pinecone-request-latency-ms': '49',
                                    'x-pinecone-response-duration-ms': '51'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'example-namespace': {'vector_count': 8}},
 'storageFullness': 0.0,
 'total_vector_count': 8,
 'vector_type': 'dense'}


# Perform Semantic Search

In [49]:
# Define the query
query = "AI is changing the world"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

In [50]:
# Returns results sorted by similarity score
print(results)

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '899',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 01:54:03 GMT',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '401',
                                    'x-pinecone-api-version': '2025-10',
                                    'x-pinecone-max-indexed-lsn': '2',
                                    'x-pinecone-response-duration-ms': '403'}},
 'result': {'hits': [{'_id': '4',
                      '_score': 0.4113227128982544,
                      'fields': {'chunk_text': 'Artificial intelligence is '
                                               'transforming industries'}},
                     {'_id': '0',
                      '_score': 0.21300579607486725,
                      'fields': {'chunk_text'

In [54]:
# Print the results
for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    print(f"id: {hit['_id']:<5} | score: {round(hit['_score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 4     | score: 0.41  | text: Artificial intelligence is transforming industries
id: 0     | score: 0.21  | text: Machine learning is a subset of AI                
id: 1     | score: 0.16  | text: Deep learning uses neural networks                
id: 2     | score: 0.16  | text: Python is used for data science                   
id: 6     | score: 0.1   | text: Data engineering involves pipelines               
id: 3     | score: 0.1   | text: Cars and vehicles are transportation              
id: 7     | score: 0.07  | text: Big data tools include Hadoop and Spark           
id: 5     | score: 0.06  | text: Football is a popular sport                       


# Delete the index when done

In [55]:
pc.delete_index(index_name)